In [ ]:
# -------------------------
# Analyze MTG Collection
# -------------------------
from itables import show
from typing import List, Tuple
import pandas as pd

from pathlib import Path

from src.decorators.profiler import profile
from src.models.card_entry import CardEntry
from src.price_fetcher.fetcher import PriceFetcher
from src.index.card_index import CardIndex
from src.parser.collection_parser import CollectionParser
from src.dataframe_builder.dataframe_builder import build_collection_dataframe
from src.parser.collection_filter import CollectionFilter


ALL_COLLECTION_NAMES = [
    "MTG-FoilBoosters.txt",
    "MTG-Cards.txt",
    "BoosterPackMtgCards.txt",
    "Lands.txt",
    "MTG-Rares.txt",
    "MTG-LorwynEclipsed.txt",
    "MTG_LorwynEclipsed2.txt"
]
# The prices file is ~500 mb - fetching it fresh + profiling takes 1m47s - Doing this for an existing file takes 5s
GET_EXISTING = True # Don't actually fetch, just use one which exists on disk

# Use a cross-platform path; adjust from the notebook directory to the web_app path.
WEBAPP_REDUCED_INDEX_PATH = Path("..").resolve() / "web_app" / "prices" / "reduced_card_index.json"

# @profile # Comment to disable
def analyze_collection(all_collection_names : List[str] = ALL_COLLECTION_NAMES) -> Tuple[pd.DataFrame, List[str]]:
    """
    Analyze the MTG card collection, fetch prices, and build a dataframe.
    
    Returns:
        Tuple containing:
        - DataFrame with collection analysis
        - List of card names with missing prices
    """

    
    # Parse collection files
    collection_parser = CollectionParser()
    collection: List[CardEntry] = []
    for collection_name in all_collection_names:
        collection.extend(collection_parser.parse(collection_name))
        
    collection = CollectionFilter.filter_duplicates(collection)

        
    # Fetch prices
    price_fetcher = PriceFetcher()
    price_file = price_fetcher.fetch_default_cards_bulk(GET_EXISTING)
    
    # Build card index for owned cards
    owned_cards = {card_entry.name for card_entry in collection}
    card_index = CardIndex.create_reduced_index(price_file, owned_cards)
    card_index.dump_index(WEBAPP_REDUCED_INDEX_PATH, True)
    # Build dataframe
    df, missing = build_collection_dataframe(collection, card_index)
    
    return df, missing


# Run collection analysis with profiling
df, missing = analyze_collection()

# Display results
print("\n" + "="*50)
print("COLLECTION ANALYSIS RESULTS")
print("="*50)

if missing:
    print(f"\nCards that are missing from the cards index ({len(missing)}):")
    for card in missing:
        print(f"  - {card}")

print(f"\nTotal collection value: ${df['total_value'].sum():.2f}")
print(f"Total cards: {df['quantity'].sum()}")
print(f"Unique cards: {len(df)}")

# Display interactive table
show(df)
